### Imports

In [3]:
import csv

def load_csv_dataset(filename, dialect='excel', delimiter = ','):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, dialect=dialect, delimiter=delimiter)
        return [row for row in reader]

    
def save_csv_dataset(filename, data, header=None):
    header = header if header else list(data[0].keys())
    with open(filename, 'w', encoding="utf-8") as f:
        writer = csv.DictWriter(f,fieldnames=header)
        writer.writeheader()
        for d in data:
            writer.writerow(d)

### Relating requirements categories and respondents

In [4]:
requirements_dataset = load_csv_dataset("../data/requirements.csv")
respondents_dataset = load_csv_dataset("../data/respondents.csv", delimiter=';')
requirements_categories = list(set([x['category'] for x in requirements_dataset]))
dataset = []
for respondent in respondents_dataset:
    categories = []
    for r in requirements_dataset:
        respondents_per_code = list(set(extract_respondents_id(r['code'])))
        if respondent['id'] in respondents_per_code:
            categories.append(r['category'])
    row = {'respondent-id': respondent['id'],
           'respondent-role': respondent['role'],
           'respondent-experience': respondent['experience'],
           'respondent-education': respondent['education']}
    
    for cat in requirements_categories:
        if cat in categories:
            row[cat] = 1
        else:
            row[cat] = 0
            
    dataset.append(row)
save_csv_dataset('../data/respondents-per-requirement-category.csv', dataset)

FileNotFoundError: [Errno 2] No such file or directory: '../data/codes-report-exported-from-atlasti.txt'

### 3d Scatter Plots

In [4]:
!pip install plotly
import pandas as pd
import plotly.express as px
df = pd.read_csv("../data/respondents-per-requirement-category.csv")

categories = df.keys()[-7:]

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    
    fig = px.scatter_3d(d, x='respondent-role', y='respondent-experience', z='respondent-education',
                        labels={
                         "respondent-role": "Role",
                         "respondent-experience": "Experience",
                         "respondent-education": "Education Level"
                     }, size = d['n-respondents'], title=f"Number of respondents, per profile - {i}")
    
    fig.write_html(f"../data/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

In [5]:
!pip install plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
df = pd.read_csv("../data/respondents-per-requirement-category.csv")

categories = df.keys()[-7:]

roles = ['Phd Student', 'Student', 'Researcher', 'Software Developer', 'Customer Engineer', 'Quality Assurance', 'Software Architect/Designer','Project/Engineering Manager']
experiences = ['less than 1 year','1 - 5 years','6 - 10 years','more than 10 years']
education = ['Unfinished bachelor', 'Bachelor','Master', 'PhD']

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    fig = go.Figure(data=go.Scatter3d(
    x=d['respondent-role'],
    y=d['respondent-experience'],
    z=d['respondent-education'],
    #text=df['country'][start:end],
    mode='markers',
    marker=dict(
        sizemode='diameter',
        sizeref=1,
        size=d['n-respondents'] * 10
        )
    ))

    fig.update_layout(scene = dict(
                    xaxis = dict(
                         backgroundcolor="rgb(200, 200, 230)",
                         gridcolor="white",
                         showbackground=True,
                         zerolinecolor="white",),
                    yaxis = dict(
                        backgroundcolor="rgb(230, 200,230)",
                        gridcolor="white",
                        showbackground=True,
                        zerolinecolor="white"),
                    zaxis = dict(
                        backgroundcolor="rgb(230, 230,200)",
                        gridcolor="white",
                        showbackground=True,
                        zerolinecolor="white",),),
                    width=400,
                    margin=dict(
                    r=10, l=10,
                    b=10, t=10)
                  )

    fig.update_xaxes(categoryorder = 'array', categoryarray=roles)
    fig.update_yaxes(categoryorder = 'array', categoryarray=experiences)
    #fig.update_zaxes(categoryorder = 'array', categoryarray=education)
    
    fig.write_html(f"../data/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

### Scatter plots with symbols

In [13]:
!pip install plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def combine_plotly_figs_to_html(plotly_figs, html_fname, include_plotlyjs='cdn', 
                                separator=None, auto_open=False):
    with open(html_fname, 'w') as f:
        f.write(plotly_figs[0].to_html(include_plotlyjs=include_plotlyjs))
        for fig in plotly_figs[1:]:
            if separator:
                f.write(separator)
            f.write(fig.to_html(full_html=False, include_plotlyjs=False))

    if auto_open:
        import pathlib, webbrowser
        uri = pathlib.Path(html_fname).absolute().as_uri()
        webbrowser.open(uri)


df = pd.read_csv("../data/respondents-per-requirement-category.csv")
categories = ['information to be provided/detailed information', 
              'information to be provided/grouped information',
              'tool usage/tool execution/workflow execution',
              'tool usage/tool execution/threshold execution',
              'tool usage/tool execution/manual execution',
              'tool usage/tool interface',
              'tool usage/customization']

roles = ['Phd Student', 'Student', 'Researcher', 'Software Developer', 'Customer Engineer', 'Quality Assurance', 'Software Architect/Designer','Project/Engineering Manager']
experiences = ['less than 1 year','1 - 5 years','6 - 10 years','more than 10 years']
education = ['Unfinished bachelor', 'Bachelor','Master', 'PhD']
figures = []

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    fig = px.scatter(d, x='respondent-role', 
                     y='respondent-experience',
                     symbol='respondent-education',
                     size='n-respondents',
                     color='respondent-education',
                     width=800, 
                     height=400,
                    opacity=0.7, 
                    title=i)
    fig.update_traces(opacity=0.7)

    figures.append(fig)
    #fig.write_html(f"../data/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

combine_plotly_figs_to_html(figures, '../data/scatter-plots/plots.html')

### information to be provided/detailed information

In [21]:
p = df.loc[df['information to be provided/detailed information'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Software Architect/Designer Master               s1_r15                                                       1
                                                  PhD                  s1_r17                                                       1
                                                                       s2_r15                                                       1
                      Software Developer          Bachelor             s1_r21                                                       1
                                                                       s1_r23                                                       1
                                                                       s1_r4                                                        1
                                                                       s2_r16                                                       1
                                                                       s5_r21                                                       1
                                                  PhD                  s2_r8                                                        1
                      Student                     Bachelor             s2_r4                                                        1
6 - 10 years          Software Architect/Designer PhD                  s5_r14                                                       1
                      Software Developer          Bachelor             s1_r25                                                       1
                                                                       s2_r21                                                       1
                                                  Master               s2_r13                                                       1
more than 10 years    Project/Engineering Manager Master               s1_r14                                                       1
                                                                       s2_r9                                                        1
                                                  Unfinished bachelor  s2_r19                                                       1
                                                                       s4_r10                                                       1
                      Software Architect/Designer Bachelor             s2_r14                                                       1
                                                                       s4_r7                                                        1
                                                  PhD                  s1_r19                                                       1
                                                                       s2_r12                                                       1
                      Software Developer          Bachelor             s2_r2                                                        1
                                                                       s5_r13                                                       1
                                                  Master               s2_r6                                                        1
                                                  Unfinished bachelor  s5_r2                                                        1

### information to be provided/grouped information

In [22]:
p = df.loc[df['information to be provided/grouped information'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Researcher                  Master               s4_r12                                                       1
                      Software Architect/Designer PhD                  s2_r15                                                       1
6 - 10 years          Software Developer          Bachelor             s4_r13                                                       1
                                                  Master               s2_r11                                                       1
more than 10 years    Project/Engineering Manager Master               s2_r9                                                        1
                                                                       s3_r10                                                       1
                                                                       s4_r5                                                        1
                                                  Unfinished bachelor  s4_r10                                                       1
                      Software Architect/Designer Bachelor             s2_r14                                                       1
                                                                       s5_r16                                                       1
                                                  PhD                  s2_r12                                                       1
                                                                       s4_r6                                                        1
                                                                       s5_r15                                                       1
                      Software Developer          Master               s2_r3                                                        1

### tool usage/tool execution/workflow execution

In [24]:
p = df.loc[df['tool usage/tool execution/workflow execution'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Software Developer          Bachelor             s3_r20                                                       1
                                                                       s5_r18                                                       1
                                                                       s5_r3                                                        1
6 - 10 years          Software Developer          Master               s3_r11                                                       1
more than 10 years    Project/Engineering Manager Master               s4_r5                                                        1
                                                                       s5_r12                                                       1
                                                  Unfinished bachelor  s3_r18                                                       1
                      Software Architect/Designer PhD                  s3_r13                                                       1
                                                                       s4_r6                                                        1
                                                                       s5_r15                                                       1
                      Software Developer          Bachelor             s5_r13                                                       1

### tool usage/tool execution/threshold execution

In [26]:
p = df.loc[df['tool usage/tool execution/threshold execution'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Software Architect/Designer Master               s1_r15                                                       1
                      Software Developer          Bachelor             s1_r21                                                       1
                                                                       s1_r4                                                        1
                                                                       s4_r11                                                       1
                                                  PhD                  s1_r13                                                       1
more than 10 years    Project/Engineering Manager Unfinished bachelor  s1_r22                                                       1
                                                                       s4_r10                                                       1
                      Software Architect/Designer Master               s1_r5                                                        1
                      Software Developer          Bachelor             s4_r2                                                        1
                                                  Master               s1_r11                                                       1
                                                                       s1_r8                                                        1
                                                  PhD                  s1_r18                                                       1
                                                  Unfinished bachelor  s1_r2                                                        1

### tool usage/tool execution/manual execution

In [27]:
p = df.loc[df['tool usage/tool execution/manual execution'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role    respondent-education respondent-id                                                 
1 - 5 years           Software Developer Bachelor             s3_r14                                                       1
                                         PhD                  s3_r9                                                        1
more than 10 years    Software Developer Bachelor             s3_r19                                                       1
                                                              s3_r3                                                        1
                                         Master               s3_r8                                                        1

### tool usage/tool interface

In [28]:
p = df.loc[df['tool usage/tool interface'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Researcher                  Master               s4_r12                                                       1
                      Software Architect/Designer PhD                  s1_r17                                                       1
                                                                       s2_r15                                                       1
                      Software Developer          Bachelor             s2_r16                                                       1
                                                                       s2_r17                                                       1
                                                                       s2_r20                                                       1
                                                  PhD                  s4_r4                                                        1
                      Student                     Bachelor             s2_r4                                                        1
6 - 10 years          Software Developer          Master               s2_r11                                                       1
more than 10 years    Project/Engineering Manager Master               s1_r14                                                       1
                                                                       s4_r5                                                        1
                                                  Unfinished bachelor  s2_r19                                                       1
                                                                       s3_r18                                                       1
                                                                       s4_r10                                                       1
                      Software Architect/Designer PhD                  s2_r12                                                       1
                                                                       s4_r6                                                        1
                      Software Developer          Bachelor             s4_r2                                                        1
                                                  Master               s4_r3                                                        1
                                                  Unfinished bachelor  s4_r1                                                        1

### tool usage/customization

In [29]:
p = df.loc[df['tool usage/customization'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

information to be provided/detailed information
respondent-experience respondent-role             respondent-education respondent-id                                                 
1 - 5 years           Software Developer          Bachelor             s1_r21                                                       1
                                                                       s1_r4                                                        1
                                                                       s2_r20                                                       1
                      Student                     Bachelor             s2_r4                                                        1
more than 10 years    Project/Engineering Manager Master               s5_r12                                                       1
                                                  Unfinished bachelor  s1_r22                                                       1
                                                                       s5_r20                                                       1
                      Software Developer          Unfinished bachelor  s1_r2                                                        1